# Regime-Based Factor Strategy - Standalone Demo
## Works in any Jupyter/Colab environment

This notebook contains all code inline, no external imports needed.

In [ ]:
# Install dependencies if needed (uncomment if in Colab)
# !pip install -q pandas numpy matplotlib seaborn scipy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
np.random.seed(42)

%matplotlib inline

## Configuration

In [ ]:
# Factor ETFs
FACTOR_ETFS = ['MTUM', 'QUAL', 'SIZE', 'VLUE', 'USMV']
BENCH = 'SPY'
CASH_ETF = 'BIL'

# Signal weights
W_MOM = 0.5
W_VAL = 0.5

# Risk-off parameters
RISK_OFF_FRACTION = 0.5
CFNAI_RISK_OFF = 0.7
TC_BPS = 5  # Transaction costs

# Regime preferences
REGIME_PREFS = {
    'Expansion': {'MTUM': 1.2, 'QUAL': 1.0, 'SIZE': 1.1, 'VLUE': 0.8, 'USMV': 0.7},
    'Peak': {'MTUM': 1.0, 'QUAL': 1.2, 'SIZE': 0.8, 'VLUE': 1.0, 'USMV': 1.1},
    'Contraction': {'MTUM': 0.7, 'QUAL': 1.3, 'SIZE': 0.6, 'VLUE': 0.8, 'USMV': 1.4},
    'Recovery': {'MTUM': 1.3, 'QUAL': 1.0, 'SIZE': 1.2, 'VLUE': 1.1, 'USMV': 0.6},
    'Unknown': {'MTUM': 1.0, 'QUAL': 1.0, 'SIZE': 1.0, 'VLUE': 1.0, 'USMV': 1.0},
}

print("✓ Configuration loaded")

## 1. Generate Synthetic Data

In [ ]:
def generate_synthetic_data(periods=168):
    """Generate synthetic monthly data"""
    
    dates = pd.date_range('2010-01-01', periods=periods, freq='M')
    tickers = FACTOR_ETFS + [BENCH, CASH_ETF]
    prices = pd.DataFrame(100.0, index=dates, columns=tickers)
    
    # Generate returns
    for i in range(1, len(dates)):
        prices.loc[dates[i], 'MTUM'] = prices.loc[dates[i-1], 'MTUM'] * (1 + np.random.normal(0.008, 0.05))
        prices.loc[dates[i], 'QUAL'] = prices.loc[dates[i-1], 'QUAL'] * (1 + np.random.normal(0.007, 0.03))
        prices.loc[dates[i], 'SIZE'] = prices.loc[dates[i-1], 'SIZE'] * (1 + np.random.normal(0.009, 0.06))
        prices.loc[dates[i], 'VLUE'] = prices.loc[dates[i-1], 'VLUE'] * (1 + np.random.normal(0.006, 0.04))
        prices.loc[dates[i], 'USMV'] = prices.loc[dates[i-1], 'USMV'] * (1 + np.random.normal(0.005, 0.02))
        prices.loc[dates[i], 'SPY'] = prices.loc[dates[i-1], 'SPY'] * (1 + np.random.normal(0.007, 0.04))
        prices.loc[dates[i], 'BIL'] = prices.loc[dates[i-1], 'BIL'] * 1.0015
    
    rets = prices.pct_change().fillna(0.0)
    
    # Generate CFNAI
    cfnai = pd.DataFrame(index=dates)
    for i, date in enumerate(dates):
        month = i % 48
        if month < 24:
            cfnai.loc[date, 'CFNAI'] = np.random.normal(0.3, 0.3)
        else:
            cfnai.loc[date, 'CFNAI'] = np.random.normal(-0.2, 0.4)
    
    cfnai['CFNAI'] = cfnai['CFNAI'].rolling(3).mean().fillna(0.0)
    
    # Generate regimes
    regimes = pd.Series('Unknown', index=dates)
    cfnai_change = cfnai['CFNAI'].diff(3)
    
    for i, date in enumerate(dates):
        c = cfnai.loc[date, 'CFNAI']
        c_chg = cfnai_change.loc[date] if date in cfnai_change.index else 0
        
        if c < -0.3:
            regimes[date] = 'Contraction'
        elif c < 0 and c_chg > 0:
            regimes[date] = 'Recovery'
        elif c > 0.2:
            regimes[date] = 'Expansion'
        elif c > 0:
            regimes[date] = 'Peak'
    
    regimes = regimes.replace('Unknown', np.nan).ffill().fillna('Expansion')
    
    return prices, rets, cfnai, regimes

prices, rets, cfnai, regimes = generate_synthetic_data()

print(f"Generated {len(prices)} months of data")
print(f"\nRegime distribution:")
print(regimes.value_counts())

## 2. Calculate Factor Signals

In [ ]:
def calculate_signals(prices, rets):
    """Calculate momentum and valuation signals"""
    
    # Momentum: 12-month return
    momentum = prices[FACTOR_ETFS].pct_change(12)
    
    # Z-score momentum
    z_mom = pd.DataFrame(index=momentum.index, columns=FACTOR_ETFS)
    for col in FACTOR_ETFS:
        rolling_mean = momentum[col].rolling(36).mean()
        rolling_std = momentum[col].rolling(36).std()
        z_mom[col] = (momentum[col] - rolling_mean) / rolling_std
    
    # Valuation
    valuation = prices[FACTOR_ETFS].pct_change(12)
    
    # Z-score valuation
    z_val = pd.DataFrame(index=valuation.index, columns=FACTOR_ETFS)
    for col in FACTOR_ETFS:
        rolling_mean = valuation[col].rolling(36).mean()
        rolling_std = valuation[col].rolling(36).std()
        z_val[col] = (valuation[col] - rolling_mean) / rolling_std
    
    return z_mom.fillna(0.0), z_val.fillna(0.0)

z_mom, z_val = calculate_signals(prices, rets)
print("✓ Signals calculated")
print("\nLatest Z-Momentum:")
print(z_mom.tail(3))

## 3. Implement Strategy Logic

In [ ]:
def backtest_strategy(version='v3'):
    """Backtest v3 or v4A strategy"""
    
    all_columns = FACTOR_ETFS + [CASH_ETF]
    w = pd.DataFrame(index=rets.index, columns=all_columns, data=0.0)
    prev_w = None
    turnovers = []
    first = True
    
    spy_mom = prices[BENCH].pct_change(12)
    
    for dt in rets.index:
        # Get signals
        zm = z_mom.loc[dt] if dt in z_mom.index else pd.Series(0.0, index=FACTOR_ETFS)
        zv = z_val.loc[dt] if dt in z_val.index else pd.Series(0.0, index=FACTOR_ETFS)
        
        # Combine signals
        if zv.isna().all():
            score_raw = zm
        elif zm.isna().all():
            score_raw = -zv
        else:
            score_raw = W_MOM * zm + W_VAL * (-zv)
        
        # Apply regime tilt
        regime = regimes.loc[dt] if dt in regimes.index else "Unknown"
        prefs = REGIME_PREFS.get(regime, REGIME_PREFS["Unknown"])
        reg_w = pd.Series([prefs[f] for f in FACTOR_ETFS], index=FACTOR_ETFS)
        score = score_raw * reg_w
        
        # Normalize
        score = score.clip(lower=0.0)
        if score.sum() == 0 or score.isna().all():
            factor_weights = pd.Series(1.0/len(FACTOR_ETFS), index=FACTOR_ETFS)
        else:
            factor_weights = score / score.sum()
        
        # Calculate risk-off
        risk_off_base = 0.0
        
        # SPY momentum check
        if dt in spy_mom.index and not pd.isna(spy_mom.loc[dt]):
            if spy_mom.loc[dt] < 0:
                risk_off_base = RISK_OFF_FRACTION
        
        # CFNAI check
        c_val = cfnai.loc[dt, "CFNAI"] if dt in cfnai.index else np.nan
        if not pd.isna(c_val) and c_val < -0.5:
            risk_off_base = max(risk_off_base, CFNAI_RISK_OFF)
        
        # v4A modification
        if version == 'v4A' and regime == 'Recovery':
            risk_off = min(risk_off_base, 0.3)  # Cap at 30%
        else:
            risk_off = risk_off_base
        
        # Portfolio weights
        w_row = pd.Series(0.0, index=all_columns)
        w_row[FACTOR_ETFS] = factor_weights * (1 - risk_off)
        w_row[CASH_ETF] = risk_off
        
        # Turnover
        if first:
            to = w_row.abs().sum()
            first = False
        else:
            to = (w_row - prev_w).abs().sum()
        
        turnovers.append(to)
        prev_w = w_row
        w.loc[dt] = w_row
    
    w = w.ffill().fillna(0.0)
    
    # Returns
    gross = (w[FACTOR_ETFS] * rets[FACTOR_ETFS]).sum(axis=1) + w[CASH_ETF] * rets[CASH_ETF]
    tc = pd.Series(turnovers, index=w.index[:len(turnovers)]) * (TC_BPS / 10000.0)
    tc = tc.reindex(gross.index).fillna(0.0)
    net = gross - tc
    cum = (1 + net).cumprod()
    
    # Stats
    cagr = cum.iloc[-1] ** (12 / len(net)) - 1
    vol = net.std() * np.sqrt(12)
    sharpe = (net.mean() / net.std()) * np.sqrt(12) if net.std() > 0 else 0
    cum_max = cum.expanding().max()
    dd = (cum - cum_max) / cum_max
    mdd = abs(dd.min())
    
    stats = {
        'Strategy': f'Regime Strategy ({version})',
        'CAGR': cagr,
        'Vol': vol,
        'Sharpe': sharpe,
        'MDD': mdd
    }
    
    return net, cum, stats, w

print("✓ Strategy functions defined")

## 4. Run Strategies

In [ ]:
# Equal-weight
rets_eq = (rets[FACTOR_ETFS] * (1/len(FACTOR_ETFS))).sum(axis=1)
cum_eq = (1 + rets_eq).cumprod()

# SPY
rets_spy = rets[BENCH]
cum_spy = (1 + rets_spy).cumprod()

# v3 and v4A
rets_v3, cum_v3, stats_v3, w_v3 = backtest_strategy('v3')
rets_v4A, cum_v4A, stats_v4A, w_v4A = backtest_strategy('v4A')

print("✓ All strategies executed")
print("\nPerformance Summary:")
print(f"v3:  CAGR={stats_v3['CAGR']:.2%}, Sharpe={stats_v3['Sharpe']:.2f}, MDD={stats_v3['MDD']:.2%}")
print(f"v4A: CAGR={stats_v4A['CAGR']:.2%}, Sharpe={stats_v4A['Sharpe']:.2f}, MDD={stats_v4A['MDD']:.2%}")
print(f"\nv4A Improvement: CAGR {(stats_v4A['CAGR']-stats_v3['CAGR']):.2%}, Sharpe {(stats_v4A['Sharpe']-stats_v3['Sharpe']):.2f}")

## 5. Visualizations

In [ ]:
# Cumulative returns
plt.figure(figsize=(14, 7))
cum_eq.plot(label="Equal-Weight", linewidth=2)
cum_v3.plot(label="v3 Strategy", linewidth=2.5)
cum_v4A.plot(label="v4A Strategy", linewidth=2.5, linestyle='--')
cum_spy.plot(label="SPY", linewidth=2, linestyle=':', alpha=0.7)

plt.title("Cumulative Returns: v3 vs v4A", fontsize=16, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cash allocation with Recovery periods highlighted
fig, ax = plt.subplots(figsize=(14, 6))
w_v3['BIL'].plot(ax=ax, label='v3 Cash', linewidth=2)
w_v4A['BIL'].plot(ax=ax, label='v4A Cash', linewidth=2, linestyle='--')

# Highlight Recovery
for i, (date, regime) in enumerate(regimes.items()):
    if regime == 'Recovery':
        next_date = regimes.index[i+1] if i < len(regimes)-1 else date
        ax.axvspan(date, next_date, color='blue', alpha=0.1)

ax.set_title("Cash Allocation: v3 vs v4A (Recovery in blue)", fontsize=16, fontweight='bold')
ax.set_ylabel("Cash Weight")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Regime timeline
fig, ax = plt.subplots(figsize=(14, 4))

regime_colors = {
    'Expansion': 'green',
    'Peak': 'orange',
    'Contraction': 'red',
    'Recovery': 'blue',
}

for i, (date, regime) in enumerate(regimes.items()):
    next_date = regimes.index[i+1] if i < len(regimes)-1 else date
    ax.axvspan(date, next_date, color=regime_colors.get(regime, 'gray'), alpha=0.3)

ax.set_title("Economic Regime Timeline", fontsize=16, fontweight='bold')
ax.set_yticks([])

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, alpha=0.3, label=regime)
                  for regime, color in regime_colors.items()]
ax.legend(handles=legend_elements, loc='upper left')
plt.tight_layout()
plt.show()

## Key Takeaways

### v4A Innovation
The v4A strategy modifies risk management during **Recovery** regimes:
- **v3**: Can go up to 70% cash during recoveries (very defensive)
- **v4A**: Caps cash at 30% during recoveries (maintains 70% factor exposure)

### When v4A Outperforms
- During Recovery regimes with improving economic data
- When markets rally from lows faster than economic indicators
- In early-stage bull markets following recessions

### Trade-offs
- **Benefit**: Higher returns during recovery periods
- **Cost**: Slightly higher drawdowns if recovery fails
- **Net**: Better risk-adjusted returns when regime detection is accurate

The blue shaded areas in the cash allocation chart show where v4A makes a difference!